# LangGraph — Intro (Full Guide)

**LangGraph** is a library from LangChain for building **stateful agentic applications** as a **graph**.

Think of an agent as a *flowchart*:
- each step is a **Node** (a unit of work — a function)
- the flow between steps is an **Edge**
- everything the steps need to share is stored in a **State**

## 1. Core Components

### State
- A dictionary (usually a `TypedDict`) that holds **all the data** your graph works with.
- It is **global context** — every node can read it, and any node can write to it.
- When a node returns a dict, those values get **merged** into the State.
- By default, writing to a key **replaces** the old value. To **accumulate** (append) instead, you use a **reducer** (see section 4).

### Node
- A **function** that does one unit of work.
- Input: the current `State`. Output: a partial dict `{"key": new_value}` that gets merged into State.
- Examples: call an LLM, run a tool, fetch data, validate output.

### Edge
- Shows the **flow / direction** between nodes.
- **Normal edge**: `node_a -> node_b` always runs `node_b` after `node_a` finishes.
- **Conditional edge**: a function that decides *which* node runs next based on the current State.

> In the diagram above every arrow is an edge, every box is a node, and the shared values (like the conversation history) live in the State.

## 2. Two Ways to Build a Graph

| | **GraphAPI** | **FunctionalAPI** |
|---|---|---|
| Style | Declarative — build with `StateGraph` object | Imperative — write plain Python functions |
| Main pieces | `add_node`, `add_edge`, `add_conditional_edges`, `compile` | `@entrypoint`, `@task` decorators |
| Flow control | Edges connect nodes | Just write normal Python code (`if/for`) |
| Best for | Explicit, inspectable, complex routing (chatbots, multi-agent) | Simple-to-mid workflows that read like normal code |

Both produce the same kind of executable graph under the hood — they differ in *how you write it*.

## 3. GraphAPI — step by step

The recipe:
1. Define a **State** `TypedDict`.
2. Create `StateGraph(State)`.
3. `add_node(name, function)` — register your work functions.
4. `add_edge(from, to)` — connect them. `START` is the entry point, `END` is the exit.
5. `compile()` → an executable `app`.
6. `app.invoke(initial_state)` → runs the whole graph and returns the final State.

In [1]:
from typing import TypedDict, Literal, Annotated

from langgraph.graph import StateGraph, START, END

In [2]:
# 1. State: the shared, global context
class State(TypedDict):
    counter: int
    log: list[str]


# 2. Nodes: functions that receive State and return partial updates
def step_one(state: State) -> dict:
    print("  step_one sees counter =", state["counter"])
    return {"counter": state["counter"] + 1, "log": ["step_one ran"]}


def step_two(state: State) -> dict:
    print("  step_two sees counter =", state["counter"])
    return {"counter": state["counter"] * 2, "log": ["step_two ran"]}


# 3. Wire the graph together
graph = StateGraph(State)
graph.add_node("step_one", step_one)
graph.add_node("step_two", step_two)
graph.add_edge(START, "step_one")   # enter at step_one
graph.add_edge("step_one", "step_two")
graph.add_edge("step_two", END)     # leave after step_two

app = graph.compile()

# 4. Run it with the initial State
final_state = app.invoke({"counter": 5, "log": []})
print("\nFINAL STATE:", final_state)

  step_one sees counter = 5
  step_two sees counter = 6

FINAL STATE: {'counter': 12, 'log': ['step_two ran']}


## 4. Reducers — appending instead of replacing

In the example above, `log` ended up with only `["step_two ran"]`.
Why? Because the default behaviour **replaces** the key each time a node writes it.

When you want to **accumulate** values (very common for message history), annotate the key with a **reducer**:

```python
messages: Annotated[list, add_messages]
```

`add_messages` is a reducer — a function that decides how to merge the new value with the old one. It *appends* new messages instead of wiping old ones. This is exactly what a chatbot needs: every node can add to the conversation without losing what came before.

In [4]:
from langgraph.graph.message import add_messages


class ChatState(TypedDict):
    # Annotated[list, add_messages] -> messages get APPENDED, never replaced
    messages: Annotated[list, add_messages]


def echo_agent(state: ChatState) -> dict:
    return {"messages": ["I am the echo agent"]}


def second_agent(state: ChatState) -> dict:
    return {"messages": ["I am the second agent"]}


chat = StateGraph(ChatState)
chat.add_node("echo_agent", echo_agent)
chat.add_node("second_agent", second_agent)
chat.add_edge(START, "echo_agent")
chat.add_edge("echo_agent", "second_agent")
chat.add_edge("second_agent", END)

chat_app = chat.compile()

result = chat_app.invoke({"messages": ["hello"]})

print("Total messages kept:", len(result["messages"]))
for i, msg in enumerate(result["messages"]):
    print(f"  [{i}] {type(msg).__name__}: {msg.content}")

Total messages kept: 3
  [0] HumanMessage: hello
  [1] HumanMessage: I am the echo agent
  [2] HumanMessage: I am the second agent


## 5. Conditional edges — branching

Agents rarely run a straight line. A **conditional edge** is a function that:
1. reads the current `State`,
2. returns the name of the next node to go to.

Here the graph loops on `increment` until the value crosses 5, then goes to `END`.

In [ ]:
class LoopState(TypedDict):
    value: int


def increment(state: LoopState) -> dict:
    return {"value": state["value"] + 1}


def route(state: LoopState) -> Literal["increment", "end"]:
    """Decision function: where do we go next?"""
    if state["value"] > 5:
        return "end"
    return "increment"


looper = StateGraph(LoopState)
looper.add_node("increment", increment)
looper.add_edge(START, "increment")
looper.add_conditional_edges(
    "increment",            # source node
    route,                   # decision function
    {"increment": "increment", "end": END},  # map return value -> destination
)

loop_app = looper.compile()

print("start value 0  ->", loop_app.invoke({"value": 0}))
print("start value 10 ->", loop_app.invoke({"value": 10}))

## 6. FunctionalAPI — graphs as plain Python

Instead of wiring nodes and edges, you write a normal function decorated with `@entrypoint` and mark reusable pieces with `@task`.

A **task** is a single unit of work. An **entrypoint** is the workflow itself.
To call a task from inside the workflow you use `.result()` — this is what lets LangGraph track execution.

Control flow is just regular Python: `if`, `for`, `while`, function calls.

In [ ]:
from langgraph.func import entrypoint, task


@task
def add_one(x: int) -> int:
    return x + 1


@task
def multiply_by_two(x: int) -> int:
    return x * 2


@entrypoint()
def workflow(x: int) -> int:
    y = add_one(x).result()        # task 1
    z = multiply_by_two(y).result()  # task 2
    return z


print("workflow(10) =", workflow.invoke(10))
print("workflow(0)  =", workflow.invoke(0))

## 7. Summary — the mental model

1. **State** = shared, global dictionary every node reads and writes.
2. **Node** = a function that does one unit of work and returns partial State updates.
3. **Edge** = the flow between nodes; can be fixed or **conditional** (decided at runtime from State).
4. **Reducer** (`Annotated[key, reducer]`) = custom merge logic; `add_messages` appends instead of replacing.
5. **GraphAPI** = explicit `StateGraph` + edges — best for complex, inspectable routing.
6. **FunctionalAPI** = `@entrypoint` + `@task` — best for readable, code-shaped workflows.
7. Both compile to an executable that you run with `app.invoke(initial_state)`.

Next step: check `chatbot.ipynb`, which uses `StateGraph` + `add_messages` to build a real chat agent.